# Intervention A — encoding the catalogue

Builds every sentence-embedding artifact Intervention A needs. Run this **once**; everything
downstream (`intervention_a_model_selection`, `..._description_variants`, `..._weight_sweep`,
`steel_thread`) reads the `.npz` files it produces and never touches a GPU encoder again.

**The embeddings are keyed by `parent_asin`, not by `item_index`.** The TF-IDF cache is keyed on
the dataset fingerprint and goes stale whenever the split changes — correctly, since its vocabulary
and IDF are *fit* on one split's warm items. A pretrained encoder fits nothing, so its output
depends only on the text. Keying by ASIN makes these artifacts **split-independent**: change the
split, add a cold population, re-draw the validation set — nothing needs re-encoding.

**Outputs** (~9 GB, `../data/embeddings/`, deliberately not in version control — regenerable from
this notebook):

| file | what |
|---|---|
| `books_<model>.npz` | `title` + `blurb`, one vector per item, for each of the six encoders |
| `movies_<model>.npz` | same for the Movies catalogue |
| `books_<model>_desc.npz` | `description` split into section groups, one vector per group |
| `books_<model>_descchunk.npz` | `description` chunked on fixed windows — the control |

**Everything is idempotent.** Each call skips work whose output already exists, so re-running after
an interruption resumes rather than restarts.

In [ ]:
%%time
import os
import sys
import time

sys.path.insert(0, "..")

from recsys import intervention_a as ia

DEVICE = "cuda:1"          # GPU 0 is shared with the desktop
EMBED_DIR = "../data/embeddings"
DATA_ROOT = ".."

# Which catalogues to encode. Books is what every experiment in this project currently uses; Movies
# is encoded so that a later cross-domain run (content.py's role indirection exists precisely to
# support Books -> Movies) never needs the GPU again. Movies costs roughly 40% of Books per model,
# so ~2-3 h across the slate -- drop it from this tuple if you only need Books.
CATALOGUES = ("movies", "books")

print(f"{'model':<22}{'repo':<46}{'dim':>6}{'batch':>7}{'remote code':>13}")
for key, spec in ia.ENCODERS.items():
    print(f"{key:<22}{spec.repo:<46}{spec.dim:>6}{spec.batch:>7}"
          f"{'yes' if spec.remote_code else '-':>13}")
print(f"\nmax_seq_length {ia.MAX_SEQ_LEN}   dtype {ia.TORCH_DTYPE}   "
      f"text roles {ia.TEXT_ROLES}   sparse roles {ia.SPARSE_ROLES}")

## Status

What already exists. Re-runnable at any point to see what is left to do.

In [ ]:
%%time
def status():
    rows = []
    for key in ia.ENCODERS:
        for cat in ia.CATALOGUES:
            p = ia.embedding_path(key, cat, EMBED_DIR)
            rows.append((f"{cat}/{key}", p))
        for label, fn in (("desc-sections", ia.description_path),
                          ("desc-chunked", ia.chunked_path)):
            rows.append((f"books/{key} [{label}]", fn(key, "books", EMBED_DIR)))
    print(f"{'artifact':<44}{'size':>9}  status")
    total = 0
    for name, p in rows:
        ok = os.path.exists(p)
        gb = os.path.getsize(p) / 1e9 if ok else 0.0
        total += gb
        print(f"{name:<44}{gb:>8.2f}G  {'done' if ok else '-'}")
    print(f"{'TOTAL':<44}{total:>8.2f}G")


status()

## 1. `title` + `blurb` — the six-encoder slate

The block every variant uses. ~240 tokens per item on average (p99 813), so a 1,024-token cap
truncates 0.26% of items and leaves top-10 neighbour lists identical to an 8,192-token reference —
the cap is there for cross-model comparability, not for speed.

**One model at a time, in a fresh process, is the safe pattern** — a CUDA OOM or a device-side
assert poisons the context for everything after it, so a single bad model would otherwise take the
whole multi-hour pass down with it. In a notebook the equivalent discipline is: if a cell below
raises, **restart the kernel** before re-running it, rather than pressing on.

Movies first per model: it is ~40% the size, so a model that is going to fail fails in minutes
rather than after half an hour.

In [ ]:
%%time
for key in ia.ENCODERS:
    for catalogue in CATALOGUES:
        t0 = time.perf_counter()
        ia.encode_catalogue(key, catalogue, embed_dir=EMBED_DIR, data_root=DATA_ROOT,
                            device=DEVICE, verbose=True)
        print(f"[encode] {key}/{catalogue} in {(time.perf_counter() - t0) / 60:.1f} min\n", flush=True)

## 2. `description` — section-split, one vector per section group

Books `description` is not one document: it is Amazon's editorial sections concatenated. Measured
over all 487,790 items, **99.5%** of populated descriptions carry a recognised heading, and among
the items long enough to be truncated at 1,024 tokens, **99.6%** do — so the structure exists
exactly where it is needed. Splitting there means every chunk is a coherent unit (one complete
bio, one complete review) rather than an arbitrary slice.

Only the winning encoder is run here. The slate comparison is settled by section 1's artifacts;
these are for the description experiments and the steel thread.

In [ ]:
%%time
DESCRIPTION_MODELS = ["arctic-l-v2.0"]      # extend if another encoder needs the description blocks

for key in DESCRIPTION_MODELS:
    t0 = time.perf_counter()
    ia.encode_description_sections(key, "books", embed_dir=EMBED_DIR, data_root=DATA_ROOT,
                                   device=DEVICE, verbose=True)
    print(f"[encode] {key}/books description sections in "
          f"{(time.perf_counter() - t0) / 60:.1f} min\n", flush=True)

## 3. `description` — fixed-window chunking (the control)

The same text, cut on arbitrary 750-word windows instead of section boundaries, pooled to one
vector. Same width and same weight as the pooled section variant, so comparing the two isolates
exactly one question: **does it matter that a chunk is a coherent passage?**

It is also the naive default most pipelines use, which makes it the honest baseline for whether
any of the section-splitting work earned its complexity.

In [ ]:
%%time
for key in DESCRIPTION_MODELS:
    t0 = time.perf_counter()
    ia.encode_description_chunked(key, "books", embed_dir=EMBED_DIR, data_root=DATA_ROOT,
                                  device=DEVICE, verbose=True)
    print(f"[encode] {key}/books description chunked in "
          f"{(time.perf_counter() - t0) / 60:.1f} min\n", flush=True)

## Final status

In [ ]:
%%time
status()